<a href="https://colab.research.google.com/github/EliasNoorzad/spacecraft-pose-estimation/blob/main/notebooks/yolo_training_10k_HST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DLVS3-HST — YOLOv8s-Pose training (10K)

Train YOLOv8s-Pose on the 10,000-image dataset (8,000 train / 1,000 validation / 1,000 held-out test).

Drive organization: the converted dataset in `MyDrive/HST/hst_yolo_pose_10k/` and checkpoints in `MyDrive/HST/HST_YOLO_Runs/`.


In [1]:
!pip -q install -U ultralytics pyyaml

## 1. Mount Drive and check the dataset

In [3]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount('/content/drive')

DRIVE_ZIP = Path('/content/drive/MyDrive/HST/hst_yolo_pose_10k.zip')
LOCAL_DATASET = Path('/content/hst_yolo_pose_10k')

RUNS_DIR = Path('/content/drive/MyDrive/HST/HST_YOLO_Runs')
RUN_NAME = 'hst_pose_10k_sigma_010'

assert torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: NVIDIA L4


## 2. Copy the dataset to Colab storage

Reading training images from local Colab storage is faster than reading each image from mounted Drive. Training checkpoints will be saved to Drive.

In [4]:
import shutil
from zipfile import ZipFile

LOCAL_ZIP = Path('/content/hst_yolo_pose_10k.zip')

# Copy the ZIP from Google Drive to Colab
shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)

# Extract the dataset locally
with ZipFile(LOCAL_ZIP, 'r') as zf:
    zf.extractall('/content')

# Remove the local ZIP
LOCAL_ZIP.unlink()

print('Dataset ready:', LOCAL_DATASET)

Dataset ready: /content/hst_yolo_pose_10k


## 3. Configure YOLOv8s-Pose

Using the same architecture and training settings as the earlier 1K smoke test, including the experimental OKS sigma of 0.10 for each of the 37 keypoints.

In [5]:
import yaml
from ultralytics import YOLO
import ultralytics

config_path = LOCAL_DATASET / 'data.yaml'
with config_path.open('r') as f:
    config = yaml.safe_load(f)

config['path'] = str(LOCAL_DATASET)
config['kpt_shape'] = [37, 3]
config['kpt_oks_sigmas'] = [0.10] * 37

with config_path.open('w') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Ultralytics:', ultralytics.__version__)
print('Training config:', config_path)
print('Checkpoints:', RUNS_DIR / RUN_NAME)
model = YOLO('yolov8s-pose.pt')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics: 8.4.216
Training config: /content/hst_yolo_pose_10k/data.yaml
Checkpoints: /content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010


## 4. Train

Train on 8,000 images and validate on 1,000. The separate 1,000-image test set remains untouched until evaluation.

In [6]:
results = model.train(
    data=str(config_path),
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=25,
    seed=42,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    save=True,
    plots=True,
)

print('Training output:', model.trainer.save_dir)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hst_yolo_pose_10k/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-pose.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=hst_pose_10k_sigma_010, nbs=64, nms=None, ops

## 5. Verify the trained checkpoint

This cell only checks the saved weights. Test-set evaluation and PnP evaluation belong in separate notebooks.

In [7]:
run_dir = Path(model.trainer.save_dir)
best = run_dir / 'weights' / 'best.pt'
last = run_dir / 'weights' / 'last.pt'

assert best.is_file(), f'Best checkpoint not found: {best}'
print('Best checkpoint:', best)
print('Last checkpoint:', last if last.is_file() else 'Not found')
print('Training results:', run_dir)

Best checkpoint: /content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010/weights/best.pt
Last checkpoint: /content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010/weights/last.pt
Training results: /content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010
